In [27]:
from pyspark.sql import *
# from pyspark.sql.functions import (
#     col, when, sum, avg, count,
#     max, min, desc, asc
# )

import os
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"


spark = SparkSession.builder \
    .appName("Read CSV Example") \
    .getOrCreate()

spark


In [9]:
df = spark.read\
        .option("header","true")\
        .option("inferSchema","true")\
        .csv("datasets/loan-test.csv")

df.show(5)

+--------+------+-------+----------+------------+-------------+---------------+-----------------+----------+----------------+--------------+-------------+
| Loan_ID|Gender|Married|Dependents|   Education|Self_Employed|ApplicantIncome|CoapplicantIncome|LoanAmount|Loan_Amount_Term|Credit_History|Property_Area|
+--------+------+-------+----------+------------+-------------+---------------+-----------------+----------+----------------+--------------+-------------+
|LP001015|  Male|    Yes|         0|    Graduate|           No|           5720|                0|       110|             360|             1|        Urban|
|LP001022|  Male|    Yes|         1|    Graduate|           No|           3076|             1500|       126|             360|             1|        Urban|
|LP001031|  Male|    Yes|         2|    Graduate|           No|           5000|             1800|       208|             360|             1|        Urban|
|LP001035|  Male|    Yes|         2|    Graduate|           No|       

In [12]:
df.count()

367

In [13]:
df.columns

['Loan_ID',
 'Gender',
 'Married',
 'Dependents',
 'Education',
 'Self_Employed',
 'ApplicantIncome',
 'CoapplicantIncome',
 'LoanAmount',
 'Loan_Amount_Term',
 'Credit_History',
 'Property_Area']

### crafting a dataframe demo

In [20]:
from pyspark import *
from pyspark.sql import Row

Student = Row("firstName", "lastName","age","telephone")
s1 = Student('David','Julian',22,1000090)
s2 = Student('Mark','Webb',23,474633)
StudentData=[s1,s2]
sdf = spark.createDataFrame(StudentData)
sdf.show()
print('*********Schema*********')
sdf.printSchema()

+---------+--------+---+---------+
|firstName|lastName|age|telephone|
+---------+--------+---+---------+
|    David|  Julian| 22|  1000090|
|     Mark|    Webb| 23|   474633|
+---------+--------+---+---------+

*********Schema*********
root
 |-- firstName: string (nullable = true)
 |-- lastName: string (nullable = true)
 |-- age: long (nullable = true)
 |-- telephone: long (nullable = true)



#### Using Python list

In [21]:
student_data = [
    (1, "Amit", 20, "Computer Science", 85),
    (2, "Neha", 22, "Mechanical", 78),
    (3, "Rahul", 21, "Electrical", 90),
    (4, "Priya", 23, "Civil", 72),
    (5, "Sneha", 20, "Computer Science", 95)
]

columns = ["student_id","name","age","department","marks"]

student_df = spark.createDataFrame(student_data,columns)

In [22]:
student_df.show()
student_df.printSchema()

+----------+-----+---+----------------+-----+
|student_id| name|age|      department|marks|
+----------+-----+---+----------------+-----+
|         1| Amit| 20|Computer Science|   85|
|         2| Neha| 22|      Mechanical|   78|
|         3|Rahul| 21|      Electrical|   90|
|         4|Priya| 23|           Civil|   72|
|         5|Sneha| 20|Computer Science|   95|
+----------+-----+---+----------------+-----+

root
 |-- student_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- age: long (nullable = true)
 |-- department: string (nullable = true)
 |-- marks: long (nullable = true)



In [25]:
student_df.groupBy('department').agg(avg("marks")).alias("avg_marks").show()

+----------------+----------+
|      department|avg(marks)|
+----------------+----------+
|Computer Science|      90.0|
|      Mechanical|      78.0|
|      Electrical|      90.0|
|           Civil|      72.0|
+----------------+----------+



In [28]:
student_df.orderBy(desc("marks")).show()

+----------+-----+---+----------------+-----+
|student_id| name|age|      department|marks|
+----------+-----+---+----------------+-----+
|         5|Sneha| 20|Computer Science|   95|
|         3|Rahul| 21|      Electrical|   90|
|         1| Amit| 20|Computer Science|   85|
|         2| Neha| 22|      Mechanical|   78|
|         4|Priya| 23|           Civil|   72|
+----------+-----+---+----------------+-----+



#### create datafreme using StructType schema (this is the recommended data engineering approach).

In [33]:
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType
)

#Create Employee Data
employee_data = [
    (101, "Amit", "IT", 60000, "Pune"),
    (102, "Neha", "HR", 45000, "Mumbai"),
    (103, "Rahul", "Finance", 70000, "Delhi"),
    (104, "Priya", "IT", 80000, "Bangalore"),
    (105, "Sneha", "HR", 50000, "Pune")
]

#Define schema using StructType

employee_schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("department", StringType(), True),
    StructField("salary", IntegerType(), True),
    StructField("city", StringType(), True)
])

#creaet DataFrame
emp_df = spark.createDataFrame(employee_data,employee_schema)
emp_df.printSchema()
emp_df.show()



root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- city: string (nullable = true)

+------+-----+----------+------+---------+
|emp_id| name|department|salary|     city|
+------+-----+----------+------+---------+
|   101| Amit|        IT| 60000|     Pune|
|   102| Neha|        HR| 45000|   Mumbai|
|   103|Rahul|   Finance| 70000|    Delhi|
|   104|Priya|        IT| 80000|Bangalore|
|   105|Sneha|        HR| 50000|     Pune|
+------+-----+----------+------+---------+



In [35]:
emp_df.orderBy("department").show()

+------+-----+----------+------+---------+
|emp_id| name|department|salary|     city|
+------+-----+----------+------+---------+
|   103|Rahul|   Finance| 70000|    Delhi|
|   102| Neha|        HR| 45000|   Mumbai|
|   105|Sneha|        HR| 50000|     Pune|
|   101| Amit|        IT| 60000|     Pune|
|   104|Priya|        IT| 80000|Bangalore|
+------+-----+----------+------+---------+

